# 🤖 robotmem — Robot Experience Memory in 5 Minutes

**Your robot shouldn't start from scratch every time.**

This notebook demonstrates how robotmem gives robots persistent memory across episodes.

- **Phase A**: Random policy (no memory) → ~0% success
- **Phase B**: Random policy + write memories → ~10% success
- **Phase C**: Recall memories → guided policy → **100% success**

Zero dependencies. Runs in < 2 minutes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/robotmem/robotmem/blob/main/examples/colab_demo.ipynb)
[![GitHub](https://img.shields.io/github/stars/robotmem/robotmem?style=social)](https://github.com/robotmem/robotmem)

In [ ]:
%%capture
# Install robotmem (~45 seconds)
!pip install -q git+https://github.com/robotmem/robotmem.git

In [ ]:
import json
import math
import os
import random
import shutil

from robotmem.sdk import RobotMemory

# Reproducible
SEED = 42
random.seed(SEED)

# Isolated DB for this demo
DB_DIR = "/tmp/robotmem-colab-demo"
if os.path.exists(DB_DIR):
    shutil.rmtree(DB_DIR)
os.makedirs(DB_DIR)
DB_PATH = os.path.join(DB_DIR, "memory.db")

print("robotmem imported OK")

## 1. The Environment

A mock 2D environment with a **hidden optimal action** `[0.3, -0.5]`.

Each episode: 10 steps. Success = average action within distance 0.35 of optimal.

Random policy success rate: ~0-15%. With memory: ~100%.

In [ ]:
class SmartMockEnv:
    """Mock environment with a hidden optimal strategy."""
    OPTIMAL_ACTION = [0.3, -0.5]
    SUCCESS_RADIUS = 0.35
    STEPS_PER_EP = 10

    def __init__(self):
        self._step = 0
        self._actions = []

    def reset(self):
        self._step = 0
        self._actions = []
        return {"observation": [random.uniform(-1, 1) for _ in range(4)]}

    def step(self, action):
        self._step += 1
        self._actions.append(list(action[:2]))
        dist = math.sqrt(sum((a - o) ** 2 for a, o in zip(action, self.OPTIMAL_ACTION)))
        reward = max(0.0, 1.0 - dist)
        done = self._step >= self.STEPS_PER_EP
        success = False
        if done and self._actions:
            avg = [sum(a[i] for a in self._actions) / len(self._actions) for i in range(2)]
            avg_dist = math.sqrt(sum((a - o) ** 2 for a, o in zip(avg, self.OPTIMAL_ACTION)))
            success = avg_dist < self.SUCCESS_RADIUS
        return {"observation": [random.uniform(-1, 1) for _ in range(4)]}, reward, done, {"is_success": success}

print(f"Hidden optimal action: {SmartMockEnv.OPTIMAL_ACTION}")
print(f"Success radius: {SmartMockEnv.SUCCESS_RADIUS}")

## 2. Phase A — Baseline (No Memory)

Random actions. The robot has no idea what works.

In [ ]:
EPISODES = 30
env = SmartMockEnv()

# Phase A: random policy, no memory
successes_a = 0
for ep in range(EPISODES):
    env.reset()
    for _ in range(SmartMockEnv.STEPS_PER_EP):
        action = [random.uniform(-1, 1) for _ in range(2)]
        _, _, done, info = env.step(action)
        if done:
            successes_a += int(info["is_success"])
            break

rate_a = successes_a / EPISODES
print(f"Phase A (baseline): {rate_a:.0%} success ({successes_a}/{EPISODES})")

## 3. Phase B — Write Memories

Still random actions, but now we **learn from each episode** using `robotmem.learn()`.

The robot records what actions it tried and whether they worked.

In [ ]:
mem = RobotMemory(db_path=DB_PATH, embed_backend="onnx")

successes_b = 0
with mem.session(context={"task": "smart_mock", "phase": "B"}) as sid:
    for ep in range(EPISODES):
        env.reset()
        trajectory = []
        for _ in range(SmartMockEnv.STEPS_PER_EP):
            action = [random.uniform(-1, 1) for _ in range(2)]
            _, reward, done, info = env.step(action)
            trajectory.append(action)
            if done:
                success = info["is_success"]
                successes_b += int(success)
                # Learn from this episode
                avg_action = [sum(a[i] for a in trajectory) / len(trajectory) for i in range(2)]
                mem.learn(
                    insight=f"Episode {ep}: {'success' if success else 'fail'}, reward={reward:.2f}",
                    context={
                        "params": {"avg_action": {"value": avg_action, "type": "vector"}},
                        "task": {"success": success, "name": "smart_mock"},
                    },
                    session_id=sid,
                )
                break

rate_b = successes_b / EPISODES
print(f"Phase B (write memory): {rate_b:.0%} success ({successes_b}/{EPISODES})")
print(f"Memories stored: {successes_b + (EPISODES - successes_b)} episodes recorded")

## 4. Phase C — Recall & Guide

Now the robot **recalls successful experiences** and uses them to guide its actions.

```
recall("successful strategy", context_filter={"task.success": True})
→ returns past successful episodes with their action parameters
→ robot follows recalled actions + small noise for exploration
```

In [ ]:
def extract_guided_action(tips, dim=2):
    """Extract average action from recalled successful memories."""
    actions = []
    for tip in tips:
        ctx = tip.get("context")
        if isinstance(ctx, str):
            try:
                ctx = json.loads(ctx)
            except (json.JSONDecodeError, TypeError):
                continue
        if not isinstance(ctx, dict):
            continue
        avg_act = ctx.get("params", {}).get("avg_action", {}).get("value")
        if isinstance(avg_act, list) and len(avg_act) >= dim:
            actions.append(avg_act[:dim])
    if not actions:
        return None
    return [sum(a[i] for a in actions) / len(actions) for i in range(dim)]


successes_c = 0
for ep in range(EPISODES):
    env.reset()
    # Recall successful experiences
    tips = mem.recall(
        "successful episode strategy optimal action",
        n=5,
        context_filter={"task.success": True},
    )
    guided_action = extract_guided_action(tips)

    for _ in range(SmartMockEnv.STEPS_PER_EP):
        if guided_action:
            # Memory-guided: follow recalled action + exploration noise
            action = [v + random.gauss(0, 0.1) for v in guided_action]
        else:
            action = [random.uniform(-1, 1) for _ in range(2)]
        _, _, done, info = env.step(action)
        if done:
            successes_c += int(info["is_success"])
            break

rate_c = successes_c / EPISODES
print(f"Phase C (recall-guided): {rate_c:.0%} success ({successes_c}/{EPISODES})")
print(f"\nImprovement: {rate_a:.0%} → {rate_c:.0%} (+{rate_c - rate_a:.0%})")

mem.close()

## 5. Results

In [ ]:
import matplotlib.pyplot as plt

phases = ["Phase A\n(Baseline)", "Phase B\n(Write Memory)", "Phase C\n(Recall-Guided)"]
rates = [rate_a * 100, rate_b * 100, rate_c * 100]
colors = ["#e74c3c", "#f39c12", "#2ecc71"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(phases, rates, color=colors, width=0.6, edgecolor="white", linewidth=2)

# Add value labels
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f"{rate:.0f}%", ha="center", va="bottom", fontsize=16, fontweight="bold")

ax.set_ylim(0, 120)
ax.set_ylabel("Success Rate (%)", fontsize=13)
ax.set_title("robotmem: Robot Learns from Experience", fontsize=15, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)

# Annotation — use percentage values (rates[]) not decimals (rate_)
delta_pct = rates[2] - rates[0]
if delta_pct > 0:
    ax.annotate(f"+{delta_pct:.0f}%", xy=(2, rates[2]), xytext=(2.35, 75),
                fontsize=20, fontweight="bold", color="#2ecc71",
                arrowprops=dict(arrowstyle="->", color="#2ecc71", lw=2))

plt.tight_layout()
plt.savefig("robotmem_result.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: robotmem_result.png")

## 6. What's Inside the Memory?

Let's peek at what the robot actually remembered:

In [ ]:
mem = RobotMemory(db_path=DB_PATH, embed_backend="onnx")

# Recall top 5 successful memories
tips = mem.recall("successful strategy", n=5, context_filter={"task.success": True})

print(f"Total successful memories recalled: {len(tips)}\n")
for i, tip in enumerate(tips, 1):
    ctx = tip.get("context")
    if isinstance(ctx, str):
        ctx = json.loads(ctx)
    action = ctx.get("params", {}).get("avg_action", {}).get("value", "?")
    print(f"  Memory {i}: {tip['content']}")
    print(f"            avg_action = {action}")
    print()

print(f"Hidden optimal: {SmartMockEnv.OPTIMAL_ACTION}")
print(f"The robot discovered the optimal strategy from its own experience!")

mem.close()

## 7. How It Works

```
Episode ends → robotmem.learn(insight, context)     # Store experience
                     ↓
Next episode → robotmem.recall(query, filter)       # Retrieve relevant experience
                     ↓
              Extract action parameters              # Use experience to guide policy
                     ↓
              Action = recalled_action + noise       # Execute with exploration
```

### Key Features

| Feature | Description |
|---------|-------------|
| `learn()` | Store episode experience with structured context |
| `recall()` | Semantic search + structured filters (e.g. only successes) |
| `context_filter` | Filter by nested JSON fields (`task.success`, `params.velocity`) |
| `spatial_sort` | Sort by spatial proximity (nearest object position) |
| Offline-first | ONNX embeddings, SQLite — no internet required |
| Model-agnostic | Works with any policy: heuristic, RL, foundation models |

### Real Experiment Results (MuJoCo FetchPush-v4)

| Experiment | Baseline | With Memory | Improvement |
|-----------|----------|-------------|-------------|
| FetchPush (300ep, 10 seeds) | 42% | 67% | **+25%** |
| Cross-env Push→Slide | 4% | 12% | **+8%** |

---

**Learn more**: [GitHub](https://github.com/robotmem/robotmem) · [robotmem.com](https://www.robotmem.com)

```bash
pip install git+https://github.com/robotmem/robotmem.git
```